# פרויקט לדוגמה — Student Lifestyle — סיווג עומס לחץ לימודי

**שאלת חקר:** האם דפוסי אורח חיים קשורים לרמת לחץ מדווחת של תלמידים?

**מקור:** מחברת Student Lifestyle שהועלתה. המחברת הזאת נבנתה מחדש, ללא העתקת זהויות תלמידים או פלטים. הבחנה בין קשר לבין סיבתיות, בדיקת הטיה ומתן הסבר למגבלות מדגם.

**סדר:** שאלה → מקור ומילון נתונים → EDA → איכות נתונים → X/y → Train/Test → Pipeline → אימון → CV → מבחן סופי → מסקנות.

זוהי דוגמה פדגוגית שדורשת הרצה ואימות על נתוני המקור; אין להסתמך על תוצאות שלא הופקו בפועל.

## 1. טעינת מאגר ומקור הנתונים
[עמוד Dataset ב-Kaggle](https://www.kaggle.com/datasets/steve1215rogg/student-lifestyle-dataset)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay, r2_score, mean_absolute_error
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
# התקנה/הורדה ב-Colab עם גישה לרשת. שם הקובץ נבדק לפי הנתיב, לא לפי ניחוש כתובת CSV.
%pip -q install kagglehub
import kagglehub
from pathlib import Path
folder=Path(kagglehub.dataset_download("steve1215rogg/student-lifestyle-dataset"))
files=sorted(folder.rglob("*.csv"))
print("קובצי CSV זמינים:",[p.name for p in files])
match=[p for p in files if p.name.lower()=="student_lifestyle_dataset.csv"]
if not match:
    raise FileNotFoundError("לא נמצא student_lifestyle_dataset.csv. בדקו את שמות הקבצים והגרסה שהורדה.")
df=pd.read_csv(match[0])
print("Rows, columns:",df.shape)
display(df.head())

## 2. הבנת הנתונים
בדקו שמות עמודות, סוגי נתונים, חסרים וכפילויות. השלימו בתא Markdown את הגדרת יחידת התצפית, משמעות המטרה ומקור הנתונים.

In [ ]:
print(df.dtypes)
print("חסרים:\n",df.isna().sum().sort_values(ascending=False).head(15))
print("שורות זהות:",df.duplicated().sum())
print("עמודות:",list(df.columns))

## 3. בחירת מטרה ומאפיינים
**y = Stress_Level**. מאפיינים המוצעים: Study_Hours_Per_Day, Extracurricular_Hours_Per_Day, Sleep_Hours_Per_Day, Social_Hours_Per_Day, Physical_Activity_Hours_Per_Day, GPA. אין לכלול יעד נגזר או מאפיינים המכילים בפועל את תוצאות ההישגים.

In [ ]:
target="Stress_Level"
requested=["Study_Hours_Per_Day","Extracurricular_Hours_Per_Day","Sleep_Hours_Per_Day","Social_Hours_Per_Day","Physical_Activity_Hours_Per_Day","GPA"]
needed=[target]+requested
missing=[col for col in needed if col not in df.columns]
if missing:
    raise ValueError("חסרות עמודות שנדרשות בדוגמה: "+str(missing)+". אין להמשיך בלי להתאים את המילון למאגר המקורי.")
data=df[needed].dropna(subset=[target]).copy()
X=data[requested]
y=data[target]
print("מספר רשומות עם יעד:",len(data))
print("מאפיינים:",list(X.columns))

## 4. EDA — התפלגות יעד וסוגי משתנים
נסחו שתי שאלות חקר. כל גרף חייב כותרת, תיאור ממצא ומגבלה.

In [ ]:
plt.figure(figsize=(8,4))
sns.countplot(data=data,x=target,order=data[target].value_counts().index)
plt.xticks(rotation=25)
plt.title("Target distribution")
plt.tight_layout();plt.show()
print(data[target].value_counts(dropna=False))

## 5. הכנה בלי דליפת מידע
נחלק את הנתונים **לפני** השלמה/תקנון/קידוד. Pipeline ילמד פרמטרים רק ב־Train ובקיפולי אימון של CV.

In [ ]:
numeric=X.select_dtypes(include="number").columns.tolist()
category=[c for c in X.columns if c not in numeric]
pre=ColumnTransformer([
    ("num",Pipeline([("fill",SimpleImputer(strategy="median")),("scale",StandardScaler())]),numeric),
    ("cat",Pipeline([("fill",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))]),category)
])
print("numeric:",numeric,"categorical:",category)
print('התפלגות יעד:', y.value_counts())
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
print("train:",X_train.shape,"test:",X_test.shape)

## 6. מודל בסיס והשוואה ל־KNN
השוו למסווג שבוחר תמיד במחלקה הנפוצה. אין להסתמך על Accuracy כאשר המחלקות לא מאוזנות.

In [ ]:
baseline=DummyClassifier(strategy="most_frequent")
baseline.fit(X_train,y_train)
print("Baseline Accuracy:",accuracy_score(y_test,baseline.predict(X_test)))
model=Pipeline([("prepare",pre),("knn",KNeighborsClassifier(n_neighbors=5))])
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
cv_scores=cross_val_score(model,X_train,y_train,cv=cv,scoring="f1_macro")
print("CV macro F1:",cv_scores.mean())
model.fit(X_train,y_train)
pred=model.predict(X_test)
print("Test Accuracy:",accuracy_score(y_test,pred))
print(classification_report(y_test,pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,pred,xticks_rotation=30)
plt.show()

## 7. מסקנות, מגבלות, אתיקה ורפלקציה
כתבו: (א) תשובה לשאלת החקר, (ב) שתי תובנות מבוססות גרפים, (ג) השוואה לבסיס או CV, (ד) מגבלת נתונים אחת, (ה) מדוע אין להסיק סיבתיות או להכליל למדגם אחר, (ו) מקורות ורישוי.

**הגשה:** Notebook שניתן להריץ; מילון משתנים; טבלת ניקוי; 3 גרפים עם פירוש; מדדים; תיעוד הפרדת Train/Test; סיכום מסודר.

## 8. בדיקת איכות
- האם נתוני CSV זמינים וגרסתם מתועדת?
- האם Stress_Level אינה נכללת בין המאפיינים?
- האם כל המרות הנתונים נלמדות רק על Train?
- האם המדד מחושב ב־Test שלא שימש לבחירת מודל?
- האם המודל והנתונים מתאימים לדרישות המחוון התקף?
- האם יש אישור לשימוש בנתוני המקור ולהפצתם?